## 神经网络 neural networks

### 定义网络

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 看一下 默认的 Net 是什么

class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()
        # 1个输入通道，6个输出通道，5x5的卷积核
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        # 仿射变换：y = Wx + b
        self.fc1 = nn.Linear(16 * 5 * 5, 120)  # 16*5*5 来自图像尺寸的推导，见第4节
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, input):
        # C1：卷积层，输出 6 张 28x28 的特征图，经过 relu 激活
        c1 = F.relu(self.conv1(input))
        # S2：池化层（下采样），2x2 窗口，输出 6 张 14x14 的特征图
        s2 = F.max_pool2d(c1, (2, 2))
        # C3：卷积层，输出 16 张 10x10 的特征图，经过 relu 激活
        c3 = F.relu(self.conv2(s2))
        # S4：池化层，输出 16 张 5x5 的特征图
        s4 = F.max_pool2d(c3, 2)
        # 展平成一维向量，保留 batch 维度（第6节详述）
        s4 = torch.flatten(s4, 1)
        # F5：全连接层 + relu
        f5 = F.relu(self.fc1(s4))
        # F6：全连接层 + relu
        f6 = F.relu(self.fc2(f5))
        # 输出层：10个类别的原始分数（不加 softmax，通常交给损失函数处理）
        output = self.fc3(f6)
        return output


net = Net()
print(net)




Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


### 笔记：`self` / `super` / 动态图（对初学者很重要）

**`self`**
- `self` 就是调用该方法的实例本身。`net.forward(x)` 底层等价于 `Net.forward(net, x)`——Python 会把调用者自动作为第一个参数传入，方法定义时必须显式写出这个形参来接收它。
- `__init__` 里 `self.conv1 = nn.Conv2d(...)` 把层绑定到"这个实例"上；`forward` 里 `self.conv1(input)` 从同一个实例取回来用。没有 `self`，两个方法之间就没法共享这些层。

**`super().__init__()`**
- `super(Net, self).__init__()` 是显式写法；`super().__init__()` 是 Python 3 的等价简写（编译器自动用方法所在类的隐藏 `__class__` 和第一个参数补全）。两者效果相同，但无参 `super()` 只能在类方法体内直接用。
- 这一步必须调用，因为 `nn.Module.__init__` 会初始化 `_parameters`/`_modules` 等内部字典，跳过它会导致后面 `self.conv1 = ...` 这样的子层注册失败。

**动态图（define-by-run）**
- `forward` 只是普通 Python 代码。每次调用 `net(input)`，PyTorch 都边执行边记录：`conv1(input)` → `ConvolutionBackward0`，`F.relu(...)` → `ReluBackward0`，`F.max_pool2d(...)` → `MaxPool2DWithIndicesBackward0`，`nn.Linear` 内部的矩阵乘 → `AddmmBackward0`……一路把这些节点连起来，直到各层的 `weight`/`bias`（`AccumulateGrad` 叶子节点）。
- 只要参与运算的张量里有一个 `requires_grad=True`（这里是各层权重，默认就是 `True`），输出就会带上 `grad_fn`，说明这张图已经建好了。
- 这张图是**一次性、跑一次建一次**的：`backward()` 沿图反向走一遍算出梯度后，图默认就被释放（除非 `retain_graph=True`）；下次再 `forward` 会重新构建一张新的图。这与静态图（先定义好整张图再反复喂数据）是本质区别。


In [ ]:
# 完整最小示例：创建网络 -> 造输入 -> 使用网络 -> 看输出，四步缺一不可
input = torch.randn(1, 1, 32, 32)   # ① 造一个假输入：1张图，1通道，32x32
out = net(input)                     # ② 使用网络（前向传播，同时动态建图）
print(out)                           # ③ 看输出：10个类别的原始分数

# ④ 验证动态图确实生成了：out 带有 requires_grad 和 grad_fn
print('\nout.requires_grad:', out.requires_grad)
print('out.grad_fn      :', out.grad_fn)

# 反向传播前，所有参数的梯度都是 None
print('\n反向传播前 conv1.weight.grad =', net.conv1.weight.grad)

# 沿着刚建好的动态图反向传播一次
out.sum().backward()

# 反向传播后，梯度被填充进每个参数的 .grad
print('反向传播后 conv1.weight.grad.shape =', net.conv1.weight.grad.shape)
